In [17]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [18]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [19]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [20]:
v1.dot(dv)

np.float32(0.3233239)

In [21]:
q2 = "How can I install Docker?"
v2 = model.encode(q2)

In [22]:
v2.dot(dv)

np.float32(0.06514119)

In [23]:
from ingest import load_faq_data
documents = load_faq_data()

In [24]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [25]:
from tqdm.auto import tqdm

In [27]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [28]:
import numpy as np
X = np.array(vectors)

In [29]:
X.shape

(1350, 384)

# Scoring Questions

In [30]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [31]:
score = X.dot(v_query)

In [32]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

# Best Match

Best match can be found with the highest score with the documents

In [33]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294106))

In [34]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [35]:
top5 = np.argsort(scores)[-5:]

In [40]:
top5

array([  7, 538, 907, 625,   2])

In [41]:
scores = np.array(scores)

In [42]:
scores[top5]

array([0.56009996, 0.65363127, 0.71921337, 0.7579372 , 0.76294106],
      dtype=float32)